# TP 5 — Discovery : construire un inventaire qui sait dire « je ne sais pas »

**Public cible :** étudiant, technicien, NetOps ou analyste cyber débutant. **Durée : 60 minutes.**

Tout est synthétique et hors ligne. Aucune écoute, requête, sonde ou découverte active n'est envoyée. La mission est de réconcilier des indices **passifs**, puis de décrire un choix **actif** qui resterait soumis à une autorisation distincte sur un vrai réseau.

> Règle du TP : **non découvert ne signifie pas absent**. Une adresse, un nom et un service sont des relations observées, pas une identité durable.


## Contrat d'apprentissage

À la fin du TP, vous saurez :

1. distinguer une observation passive d'une sonde active ;
2. relier MAC, IP, nom et service sans les confondre ;
3. conserver la source, la fraîcheur et un indice de confiance explicable ;
4. rendre visibles contradiction et angle mort ;
5. choisir une prochaine preuve sous budget et écrire un plan inerte.

Avant d'exécuter les cellules, prédisez : quel équipement sera ambigu, quelle relation expirera, et quel appareil restera invisible ?


In [ ]:
from collections import defaultdict
from base64 import b64encode
from html import escape
import csv
import io
import json

SEED = 20260730
NOW_S = 300
SOURCE_TTL_S = {"dhcp": 180, "arp": 60, "mdns": 120, "lldp": 120}
observations = [
    {"source": "dhcp", "seen_s": 290, "mac": "02:00:00:00:00:11", "ip": "192.0.2.11", "name": "drone-alpha", "service": None, "confidence": 0.90},
    {"source": "arp",  "seen_s": 298, "mac": "02:00:00:00:00:11", "ip": "192.0.2.11", "name": None, "service": None, "confidence": 0.85},
    {"source": "mdns", "seen_s": 295, "mac": "02:00:00:00:00:11", "ip": "192.0.2.11", "name": "alpha.local", "service": "_telemetry._udp", "confidence": 0.75},
    {"source": "dhcp", "seen_s": 100, "mac": "02:00:00:00:00:22", "ip": "192.0.2.12", "name": "relay-bravo", "service": None, "confidence": 0.90},
    {"source": "arp",  "seen_s": 282, "mac": "02:00:00:00:00:22", "ip": "192.0.2.12", "name": None, "service": None, "confidence": 0.85},
    {"source": "lldp", "seen_s": 294, "mac": "02:00:00:00:00:33", "ip": None, "name": "switch-gamma", "service": "bridge", "confidence": 0.80},
    {"source": "arp",  "seen_s": 299, "mac": "02:00:00:00:00:44", "ip": "192.0.2.12", "name": None, "service": None, "confidence": 0.85},
    {"source": "arp",  "seen_s": 289, "mac": "02:00:00:00:00:55", "ip": "192.0.2.15", "name": None, "service": None, "confidence": 0.85},
]

for item in observations:
    age = NOW_S - item["seen_s"]
    item["age_s"] = age
    item["freshness"] = round(max(0.0, 1.0 - age / SOURCE_TTL_S[item["source"]]), 2)
    item["fresh"] = age <= SOURCE_TTL_S[item["source"]]

assert len(observations) == 8
assert any(not item["fresh"] for item in observations)
print("Fixtures passives :", ", ".join(f'{source}={sum(o["source"] == source for o in observations)}' for source in SOURCE_TTL_S))


## 1. Réconciliation : ne jamais effacer le désaccord

La clé de travail est ici la MAC, mais elle n'est ni une personne ni une identité inviolable. Pour chaque attribut, on conserve la preuve la plus fraîche ; toutes les sources restent citées. L'indice de **confiance** est une heuristique pédagogique, pas une probabilité calibrée. La **fraîcheur** mesure seulement l'âge relatif à un TTL de fixture.


In [ ]:
by_mac = defaultdict(list)
for observation in observations:
    by_mac[observation["mac"]].append(observation)

def freshest_value(items, field):
    candidates = [item for item in items if item.get(field) is not None]
    return max(candidates, key=lambda item: item["seen_s"])[field] if candidates else None

assets = []
for mac, items in sorted(by_mac.items()):
    fields = {field: freshest_value(items, field) for field in ("ip", "name", "service")}
    completeness = sum(value is not None for value in fields.values()) / 3
    source_count = len({item["source"] for item in items})
    evidence_score = min(0.98, 0.35 + 0.12 * source_count + 0.25 * completeness)
    assets.append({
        "mac": mac, **fields,
        "sources": sorted({item["source"] for item in items}),
        "confidence": round(evidence_score, 2),
        "freshness": max(item["freshness"] for item in items),
        "stale_evidence": sorted(item["source"] for item in items if not item["fresh"]),
    })

fresh_ip_claims = defaultdict(set)
for item in observations:
    if item["fresh"] and item.get("ip"):
        fresh_ip_claims[item["ip"]].add(item["mac"])
contradictions = [
    {"ip": ip, "macs": sorted(macs), "kind": "une IP, plusieurs MAC fraîches"}
    for ip, macs in sorted(fresh_ip_claims.items()) if len(macs) > 1
]
blind_spots = [
    "équipement silencieux ou endormi",
    "service non annoncé ou filtré",
    "segment absent des fixtures autorisées",
]
discovery_report = {
    "seed": SEED,
    "evidence_count": len(observations),
    "asset_hypotheses": assets,
    "contradictions": contradictions,
    "blind_spots": blind_spots,
    "claim": "inventaire partiel : non découvert ne signifie pas absent",
}

assert len(assets) == 5
assert contradictions[0]["ip"] == "192.0.2.12"
assert assets[1]["stale_evidence"] == ["dhcp"]
print(f"Hypothèses={len(assets)} · contradictions={len(contradictions)} · angles morts={len(blind_spots)}")


## 2. Sonde active : maximiser l'information, pas le bruit

Une observation passive exploite ce qui existe déjà ; une sonde active crée du trafic et peut modifier la cible ou ses journaux. Sur un vrai réseau, elle exige une **autorisation** définissant périmètre, cible, débit, durée, arrêt et conservation des traces. Ici, nous ne construisons qu'un **probe_plan** textuel : aucun exécuteur, aucun socket, aucune commande.

Budget pédagogique : 3 points. Choisissez les preuves qui réduisent d'abord la contradiction et l'angle mort le plus coûteux.


In [ ]:
probe_candidates = [
    {"probe": "ARP ciblé 192.0.2.12", "cost": 1, "information_gain": 3, "radius": "une IP"},
    {"probe": "SNMP ciblé switch-gamma", "cost": 2, "information_gain": 4, "radius": "un équipement"},
    {"probe": "mDNS segment lab", "cost": 2, "information_gain": 2, "radius": "un segment"},
    {"probe": "balayage ICMP /28", "cost": 4, "information_gain": 5, "radius": "seize adresses"},
]
budget = 3
remaining = budget
selected = []
for candidate in sorted(probe_candidates, key=lambda item: (-item["information_gain"] / item["cost"], item["probe"])):
    if candidate["cost"] <= remaining:
        selected.append(candidate)
        remaining -= candidate["cost"]

probe_plan = {
    "execution_allowed": False,
    "authorization_required": True,
    "budget": budget,
    "budget_remaining": remaining,
    "selected": selected,
    "stop_condition": "arrêt après la preuve ciblée ou au premier effet inattendu",
    "note": "description pédagogique inerte ; aucun trafic réel",
}
discovery_report["probe_plan"] = probe_plan
assert probe_plan["execution_allowed"] is False
assert sum(item["cost"] for item in selected) <= budget
print("Plan inerte :", " + ".join(item["probe"] for item in selected), f"· reste={remaining}")


## 3. Tableau de preuve : voir ce que l'inventaire affirme et ce qu'il ignore

Le tableau relie chaque hypothèse à ses sources. Le rouge n'est pas « appareil hostile » : c'est une contradiction à résoudre. Les traits pointillés signalent les champs manquants.


In [ ]:
class DiscoveryDashboard:
    def __init__(self, report):
        self.report = report

    def _repr_html_(self):
        rows = []
        for index, asset in enumerate(self.report["asset_hypotheses"]):
            y = 252 + index * 50
            ip = escape(asset["ip"] or "—")
            name = escape(asset["name"] or "—")
            service = escape(asset["service"] or "—")
            sources = escape(" + ".join(asset["sources"]))
            confidence_width = int(120 * asset["confidence"])
            stale = " · périmé: " + "/".join(asset["stale_evidence"]) if asset["stale_evidence"] else ""
            color = "#ff5c68" if ip == "192.0.2.12" else "#40c4ff"
            rows.append(f'''
              <text x="48" y="{y}" fill="#dce7f5" font-size="14">{escape(asset['mac'][-8:])}</text>
              <text x="190" y="{y}" fill="{color}" font-size="14">{ip}</text>
              <text x="340" y="{y}" fill="#dce7f5" font-size="14">{name}</text>
              <text x="520" y="{y}" fill="#dce7f5" font-size="14">{service}</text>
              <text x="700" y="{y}" fill="#9fb0c6" font-size="13">{sources}{escape(stale)}</text>
              <rect x="930" y="{y - 13}" width="120" height="10" rx="5" fill="#24364d"/>
              <rect x="930" y="{y - 13}" width="{confidence_width}" height="10" rx="5" fill="#42e6a4"/>
              <text x="1065" y="{y}" fill="#dce7f5" font-size="13">{asset['confidence']:.2f}</text>
            ''')
        selected = " + ".join(escape(item["probe"]) for item in self.report["probe_plan"]["selected"])
        return f'''
        <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1160 650" role="img" aria-label="Tableau de découverte synthétique" style="width:100%;max-width:1160px;background:#071421;border-radius:20px;font-family:Inter,Arial,sans-serif">
          <rect width="1160" height="650" rx="20" fill="#071421"/>
          <text x="44" y="58" fill="#f4f7fb" font-size="30" font-weight="700">Discovery — preuves, désaccords, angles morts</text>
          <text x="44" y="88" fill="#9fb0c6" font-size="15">passif observé · actif seulement planifié · non découvert ≠ absent</text>
          <rect x="44" y="112" width="242" height="76" rx="14" fill="#12304a"/>
          <text x="64" y="142" fill="#9fb0c6" font-size="13">HYPOTHÈSES D'ACTIFS</text><text x="64" y="174" fill="#40c4ff" font-size="27" font-weight="700">{len(self.report['asset_hypotheses'])}</text>
          <rect x="304" y="112" width="242" height="76" rx="14" fill="#3d1b2a"/>
          <text x="324" y="142" fill="#ff9ca4" font-size="13">CONTRADICTION</text><text x="324" y="174" fill="#ff5c68" font-size="27" font-weight="700">192.0.2.12 × 2</text>
          <rect x="564" y="112" width="242" height="76" rx="14" fill="#17372f"/>
          <text x="584" y="142" fill="#a9dacb" font-size="13">PREUVES PASSIVES</text><text x="584" y="174" fill="#42e6a4" font-size="27" font-weight="700">{self.report['evidence_count']}</text>
          <rect x="824" y="112" width="292" height="76" rx="14" fill="#3a3016"/>
          <text x="844" y="142" fill="#e9d59b" font-size="13">PLAN ACTIF INERTE</text><text x="844" y="174" fill="#ffbd2e" font-size="23" font-weight="700">false · 0 paquet émis</text>
          <text x="48" y="222" fill="#9fb0c6" font-size="12">MAC</text><text x="190" y="222" fill="#9fb0c6" font-size="12">IP</text><text x="340" y="222" fill="#9fb0c6" font-size="12">NOM</text><text x="520" y="222" fill="#9fb0c6" font-size="12">SERVICE</text><text x="700" y="222" fill="#9fb0c6" font-size="12">SOURCES / ÉTAT</text><text x="930" y="222" fill="#9fb0c6" font-size="12">CONFIANCE</text>
          {''.join(rows)}
          <rect x="44" y="516" width="1072" height="92" rx="14" fill="#10243a"/>
          <text x="64" y="548" fill="#ffbd2e" font-size="15" font-weight="700">Prochaine preuve sous budget 3</text>
          <text x="64" y="576" fill="#dce7f5" font-size="14">{selected}</text>
          <text x="64" y="598" fill="#9fb0c6" font-size="12">Autorisation requise sur un vrai réseau · ce notebook n'émet rien.</text>
        </svg>
        '''

DiscoveryDashboard(discovery_report)


## 4. Exporter la preuve, pas une certitude

Les deux liens suivants sont des data URI produits en mémoire. Le JSON conserve les limites et le plan inerte ; le CSV facilite une revue humaine des hypothèses. Aucun fichier n'est créé par le notebook.


In [ ]:
json_payload = json.dumps(discovery_report, ensure_ascii=False, indent=2, sort_keys=True).encode("utf-8")
csv_buffer = io.StringIO(newline="")
writer = csv.DictWriter(csv_buffer, fieldnames=["mac", "ip", "name", "service", "sources", "confidence", "freshness", "stale_evidence"])
writer.writeheader()
for asset in assets:
    writer.writerow({**asset, "sources": "+".join(asset["sources"]), "stale_evidence": "+".join(asset["stale_evidence"])})
csv_payload = csv_buffer.getvalue().encode("utf-8")
json_uri = "data:application/json;base64," + b64encode(json_payload).decode("ascii")
csv_uri = "data:text/csv;base64," + b64encode(csv_payload).decode("ascii")

class DownloadLinks:
    def _repr_html_(self):
        return (f'<p><a download="discovery_report.json" href="{json_uri}">Télécharger le rapport JSON</a>'
                f' · <a download="inventaire_hypotheses.csv" href="{csv_uri}">Télécharger le CSV</a></p>')

assert json.loads(json_payload)["probe_plan"]["execution_allowed"] is False
assert csv_payload.startswith(b"mac,ip,name,service")
DownloadLinks()


## 5. Transfert, autorisation et limites

Restituez en cinq phrases : (1) ce qui est observé, (2) ce qui est réconcilié, (3) la contradiction, (4) l'angle mort, (5) la prochaine preuve et son coût. Puis remplacez le TTL ARP de 60 s par 10 s : quelles hypothèses disparaissent, et pourquoi cela ne prouve-t-il pas que les appareils sont absents ?

**Barème rapide — 10 points :** passif/actif 2 ; relations et sources 2 ; confiance/fraîcheur 2 ; contradiction/angle mort 2 ; prochaine preuve autorisée et bornée 2.

**Limites :** fixtures connues, petit segment, heuristique de confiance non calibrée, aucune identité forte, aucun scan, aucun sniff, aucun équipement réel. L'inventaire n'est ni une CMDB exhaustive, ni un audit de sécurité, ni une preuve de présence ou d'absence. Sur un réseau réel, toute acquisition ou sonde active demande une autorisation propre, un rayon d'impact et une condition d'arrêt vérifiables.
